# Cambio climático y energías renovables en Argentina
## Informe final del proyecto

**Desafío Profesional de Data Science — Digital House**  
Etapa 4 · Paso 4.5 — Storytelling final del proyecto

---

Este informe cierra un proyecto de cuatro etapas en el que exploramos, limpiamos, modelamos y comparamos una colección de datos sobre emisiones de CO₂ y generación de energía renovable, con foco en Argentina. Acá no hay código de modelado nuevo: lo que hay es la historia que cuentan los datos, traducida para un lector que quiere entender qué aprendimos y qué se puede hacer con eso. Quien busque los detalles técnicos los encuentra en los notebooks de cada etapa, listados al final.

---

## 1. Por qué este proyecto importa

El cambio climático es un problema medido. Sabemos cuánto CO₂ emite cada país por año, sabemos cuánta energía consume, sabemos qué porcentaje de esa energía viene de fuentes renovables. Lo que es más difícil es *entender qué tan acoplados están esos números entre sí, y qué los empuja en una dirección u otra*.

Este proyecto se hizo cuatro preguntas concretas:

> **P1**. ¿Cómo evolucionaron las emisiones de CO₂ a lo largo del tiempo, en el mundo y en Argentina?
>
> **P2**. ¿Qué relación hay entre las emisiones y variables como población o PBI per cápita?
>
> **P3**. ¿Qué patrones emergen en la participación de energías renovables en distintos países?
>
> **P4**. ¿Cómo se compara la situación de Argentina con otros países en emisiones y energía renovable?

El informe se organiza alrededor de esas cuatro preguntas. Cada sección sigue una misma lógica: qué dicen los datos, qué le pedimos a un modelo, qué nos dejó la respuesta y qué se puede hacer con eso.

---

## 2. Los datos que tuvimos en mano

El proyecto trabaja sobre cinco fuentes que cubren ángulos complementarios del problema:

| # | Fuente | Cobertura | Qué aporta |
|---|---|---|---|
| 1 | Emisiones eléctricas Argentina (CAMMESA) | 2007–2017, anual | Factor de emisión del sistema eléctrico argentino |
| 2 | Estadísticas energéticas mundiales (BP) | 1990–2018, anual × país | CO₂, electricidad, renovables, fósiles por país |
| 3 | PBI per cápita (Banco Mundial) | 1960–2019, anual × país | Variable económica para cruzar con emisiones |
| 4 | Población mundial (Banco Mundial) | 1960–2019, anual × país | Para calcular per cápita y normalizar |
| 5 | Generación renovable Argentina (CAMMESA) | 2011–2018, mensual × central | Detalle granular del boom renovable argentino |

**El trabajo de limpieza fue grande pero invisible.** Los datasets venían en formatos heterogéneos (Excel con encabezados desplazados, hojas mezcladas, agregaciones del Banco Mundial mezcladas con países reales, unidades distintas: MtCO₂ aquí, tCO₂/MWh allá, GWh y Mtoe en el mismo archivo). En la Etapa 2 unificamos todo y dejamos cinco datasets limpios listos para modelar — sin esa etapa, ningún modelo posterior habría funcionado. El detalle está en el notebook de limpieza, pero el aprendizaje vale citarlo: **el 60% del esfuerzo del proyecto fue limpiar datos, y eso es perfectamente normal en data science aplicada al mundo real**.

---

## 3. P1 — ¿Cómo evolucionaron las emisiones de CO₂?

### 🌍 El mundo

La historia global de las emisiones de CO₂ entre 1990 y 2018 tiene tres movimientos:

1. **1990–2000**: crecimiento moderado liderado por las economías ya desarrolladas.
2. **2000–2014**: aceleración fuerte impulsada por China, India y los países emergentes asiáticos.
3. **2014–2018**: desaceleración relativa, con varios países desarrollados estabilizando o reduciendo sus emisiones por primera vez.

Esa última fase es la más interesante. Es el primer período de la era industrial donde algunos países demostraron que **se puede crecer económicamente y bajar emisiones simultáneamente** —un fenómeno que los economistas llaman *desacople absoluto*—. Reino Unido y Portugal son los ejemplos más claros en nuestros datos.

### 🇦🇷 Argentina

Argentina sigue una trayectoria propia: emisiones que aumentaron de manera sostenida pero más lenta que el promedio mundial. En 1990 emitía ~101 MtCO₂; en 2018, ~185 MtCO₂. Es un crecimiento de 83% en 28 años, comparable al crecimiento poblacional del país, lo que sugiere que **las emisiones per cápita argentinas se mantuvieron relativamente estables** —algo que retomamos en P4—.

### 🤖 Lo que pudo hacer un modelo

Para forecasting de generación renovable mensual argentina (Pasos 4.2 y 4.3), una LSTM simple le ganó a ARIMA por unos 6 puntos de MAPE. Pero el dato que mejor enseña es el que sale del cierre comparativo del 4.4: con menos de 100 puntos en la serie, el techo de mejora vía tuning era bajísimo. **La LSTM tuneada, con todo el esfuerzo que se le puso, empató estadísticamente a la LSTM simple.** Lo dejamos como aprendizaje: no todo problema admite ser "resuelto mejor con más sofisticación".

### ✅ Recomendación P1

Para monitorear emisiones y proyectar tendencias en horizontes cortos, modelos estadísticos clásicos (ARIMA y similares) son suficientes y más fáciles de auditar. Si en el futuro se acumulan series mensuales largas (>10 años de datos), abrir el camino a modelos LSTM puede pagar — pero hoy, con la ventana disponible, no es la prioridad.

---

## 4. P2 — ¿Qué explica las emisiones per cápita?

### 🔍 El hallazgo central del proyecto

Si tuviéramos que destilar el proyecto en un solo descubrimiento, sería este:

> **Las emisiones de CO₂ per cápita están casi perfectamente explicadas por el consumo de energía per cápita.**
>
> Correlación: **ρ ≈ 0.99**.

Ese coeficiente es altísimo. Significa que conociendo cuánta energía consume una persona promedio en un país, podemos predecir con mucha precisión cuántas toneladas de CO₂ emite. Y eso tiene una lectura política directa: **bajar emisiones per cápita pasa por dos caminos posibles —consumir menos energía, o consumir energía de fuentes que no emitan—**. No hay un tercero.

### 🤖 Lo que enseñó el modelo

Cuando entrenamos un MLP (red neuronal multicapa) sobre el dataset global con seis features —población, PBI per cápita, electricidad, energía total, % de renovables, % de eólica+solar—, el modelo logró un R² de 0.93. Los modelos lineales (Ridge, Lasso, Regresión Múltiple) se quedaban en R² ≈ 0.76.

**Pero el MLP no descubrió nada mágico**. Lo que hizo fue, internamente, *construir el cociente* `energía total / población` —es decir, energía per cápita— y darse cuenta de que esa variable derivada explicaba casi todo. Cuando agregamos esa misma variable a Ridge como columna calculada, Ridge alcanzó la performance del MLP.

Esto deja una lección dorada para cualquier futuro analista: **la diferencia entre Deep Learning y modelos clásicos suele ser feature engineering implícito, no inteligencia adicional**. Si uno sabe qué buscar, los modelos clásicos compiten cabeza a cabeza. Si no, las redes neuronales pueden encontrar el cociente por uno — pagándolo en interpretabilidad y costo computacional.

### ✅ Recomendación P2

Para modelar emisiones de CO₂ a nivel país, la variable más predictiva es la energía per cápita. Cualquier política que apunte a reducir emisiones tiene que tocar uno de los dos componentes de ese cociente: el numerador (consumo total) o cambiar la matriz de generación de ese consumo (renovables vs fósil).

---

## 5. P3 — ¿Qué patrones emergen en renovables?

### 🇦🇷 Argentina: el boom RenovAr

La generación renovable mensual de Argentina —medida por las renovables "nuevas" que define la ley 27.191— pasó de ser prácticamente nula en 2011 a un crecimiento explosivo en 2018:

- **2011**: la generación renovable cubierta por la ley apenas existía (algunos parques eólicos pioneros y poco más).
- **2017**: ~2% de la demanda eléctrica del país.
- **2018**: ~2.5%, con una pendiente de crecimiento que se empinó fuerte en la segunda mitad del año por la entrada en operación de varios parques eólicos del programa RenovAr.
- **2019**: ~5%, duplicando el año anterior.

### 🌐 Una aclaración importante

Si uno mira las estadísticas globales del Banco Mundial, Argentina aparece con ~30% de generación renovable. Esa cifra es **correcta pero engañosa**: incluye las grandes represas hidroeléctricas (El Chocón, Yacyretá, Salto Grande), que están en operación desde hace décadas y son tecnológicamente antiguas.

La ley 27.191, que define la política argentina de renovables modernas, **excluye explícitamente la hidroeléctrica de más de 50 MW**. Por eso conviven dos números en los reportes:

| Métrica | 2018 |
|---|---|
| % renovable global (incluye hidro grande) | ~30.7% |
| % renovable según ley 27.191 (sin hidro grande) | ~2.5% |

Cualquier conversación sobre "Argentina y las renovables" tiene que aclarar primero de cuál de las dos cifras se está hablando. **No es un detalle técnico: es una decisión de política pública sobre qué consideramos avance en transición energética**.

### 🌍 El mundo

A nivel global, los líderes en participación renovable en 2018 eran Portugal (52%), Chile (47%) y Reino Unido (34%). En el otro extremo, Kuwait (0%), Iran (5%) y Corea del Sur (4%) tenían matrices casi totalmente fósiles. La heterogeneidad es enorme y **se explica más por geografía y política que por ingreso**: Portugal y Chile no son países más ricos que Corea del Sur, pero sí tienen condiciones físicas (sol, viento, hidrología) y marcos regulatorios que privilegiaron la transición.

### ✅ Recomendación P3

Para la conversación pública argentina sobre renovables, conviene **separar siempre el dato de hidroeléctrica grande del de renovables modernas**. Mezclarlos da la impresión equivocada de que el país ya está en una transición avanzada, cuando en realidad las renovables nuevas todavía representan una fracción muy chica de la matriz.

---

## 6. P4 — ¿Cómo se ubica Argentina en el mapa global?

Para responder esta pregunta armamos un mapa comparativo. Cada país en el dataset es un punto en un plano definido por sus emisiones per cápita y su porcentaje de renovables; el tamaño del punto refleja el PBI per cápita.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = '/content/drive/MyDrive/desafio_profesional/datos_limpios/'

df_global = pd.read_csv(DATA_PATH + 'dataset_global.csv')
df_2018 = df_global[df_global['anio'] == 2018].dropna(
    subset=['co2_per_capita', 'share_renewables', 'pbi_per_capita']
).copy()

fig, ax = plt.subplots(figsize=(13, 7.5))

es_argentina = df_2018['pais'] == 'Argentina'
otros = df_2018[~es_argentina]
arg = df_2018[es_argentina]

# Tamaño proporcional a PBI per cápita
tamano = lambda pbi: (pbi / 1000) * 8 + 30

# Otros países
ax.scatter(
    otros['share_renewables'], otros['co2_per_capita'],
    s=tamano(otros['pbi_per_capita']),
    alpha=0.55, c='#7F77DD', edgecolor='black', linewidth=0.5,
    label='Otros países', zorder=2
)

# Argentina destacada
ax.scatter(
    arg['share_renewables'], arg['co2_per_capita'],
    s=tamano(arg['pbi_per_capita']) * 1.4,
    alpha=0.95, c='#D85A30', edgecolor='black', linewidth=1.5,
    label='Argentina', zorder=4
)

# Etiquetas
for _, row in df_2018.iterrows():
    color = '#993C1D' if row['pais'] == 'Argentina' else '#3C3489'
    weight = 'bold' if row['pais'] == 'Argentina' else 'normal'
    ax.annotate(
        row['pais'],
        (row['share_renewables'], row['co2_per_capita']),
        xytext=(7, 5), textcoords='offset points',
        fontsize=9.5, color=color, fontweight=weight
    )

# Línea de promedio mundial CO2 per cápita aprox
promedio_co2 = df_2018['co2_per_capita'].median()
ax.axhline(promedio_co2, color='gray', linestyle=':', alpha=0.6,
           label=f'Mediana CO₂ p/c ({promedio_co2:.1f} t)')

ax.set_xlabel('Participación de renovables en el mix energético (%)\n(incluye hidroeléctrica grande)', fontsize=11)
ax.set_ylabel('Emisiones de CO₂ per cápita (toneladas)', fontsize=11)
ax.set_title('Argentina en el mapa global — 2018\nTamaño del punto = PBI per cápita',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 🔍 Lo que dice el mapa

Argentina aparece en una posición intermedia con tres rasgos distintivos:

1. **Emisiones per cápita moderadas** (~4.2 toneladas/año), por debajo de la mediana mundial. Está más cerca de Chile (4.7) y Reino Unido (5.4) que de los grandes emisores como Estados Unidos (15.6) o Australia (16.1).
2. **Participación renovable del 30%**, aunque ya aclaramos en P3 que esa cifra está dominada por la hidroeléctrica grande de los años 70 y 80. La renovable "moderna" recién empieza.
3. **PBI per cápita medio-bajo** (~12.000 USD), considerablemente menor que los países líderes en renovables modernas como Portugal o Reino Unido.

Esa combinación es informativa: Argentina no es un emisor grave en términos relativos, pero tampoco es un líder en transición. Está en lo que podríamos llamar **una zona de inercia**: la matriz energética actual emite poco gracias a infraestructura vieja (hidroeléctrica), pero el crecimiento futuro depende de decisiones de política que recién en la última década empezaron a empujar las renovables modernas.

### ✅ Recomendación P4

El benchmarking honesto de Argentina debería compararla con países de PBI per cápita similar y matriz parcialmente fósil —Chile, Portugal pre-2010, Brasil—. Compararla con Estados Unidos o Alemania para celebrar que emite menos confunde el análisis: la diferencia se explica por estructura económica, no por superioridad ambiental.

---

## 7. Lo que aprendimos sobre Machine Learning en el camino

Aunque el objetivo del proyecto eran las preguntas sobre cambio climático, en el proceso de responderlas aprendimos cosas sobre la práctica de Machine Learning que valen oro para cualquier proyecto futuro. Las cuatro más importantes:

**1. "Deep Learning gana" no es una regla, es una hipótesis a probar.** Las redes neuronales le ganaron a los modelos clásicos por amplio margen en el problema de regresión global (R² 0.93 vs 0.76), pero apenas empataron en forecasting de series temporales cortas. La diferencia tuvo todo que ver con la cantidad de datos disponibles y con la existencia de transformaciones no-lineales aprovechables.

**2. Reportar una sola corrida de una red neuronal es académicamente cuestionable.** En el Paso 4.3, distintas semillas aleatorias produjeron resultados de la LSTM que diferían en hasta 7 puntos de MAPE entre la mejor y la peor corrida. Si hubiéramos reportado solo la mejor seed, habríamos vendido un avance que no existía. La práctica honesta es entrenar al menos 3 veces y reportar media ± desvío.

**3. Feature engineering manual sigue siendo más poderoso que la fuerza bruta.** El cociente `energía total / población` no estaba en el dataset original. Cuando lo agregamos a mano, un modelo lineal de 7 parámetros igualó la performance de una red neuronal de 10.000 parámetros. La conclusión práctica: **antes de probar modelos más complejos, conviene siempre intentar features mejores**.

**4. La interpretabilidad importa más de lo que se dice.** En contextos de auditoría, regulación o política pública, los coeficientes lineales se pueden explicar a un comité en 30 segundos. Una red neuronal, no. Esa diferencia hace que en muchos contextos profesionales reales, los modelos clásicos sean *preferibles* aunque ganen menos en métrica.

---

## 8. El viaje del proyecto en una imagen

Para cerrar la narrativa, una sola visualización que conecta las cuatro etapas del proyecto: cómo pasamos de cinco archivos crudos a recomendaciones accionables.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(13, 6))
ax.set_xlim(0, 12)
ax.set_ylim(0, 8)
ax.axis('off')

# Definición de las 4 etapas
etapas = [
    {'x': 0.5, 'titulo': 'Etapa 1', 'subtitulo': 'EDA',
     'detalle': '5 fuentes\n+170k filas\nexploradas',
     'insight': 'Patrones temporales,\noutliers, brechas',
     'color': '#CECBF6', 'border': '#534AB7'},
    {'x': 3.4, 'titulo': 'Etapa 2', 'subtitulo': 'Limpieza & ETL',
     'detalle': 'Unificación,\ndedup, joins\npor país-año',
     'insight': '5 datasets\nlimpios listos',
     'color': '#9FE1CB', 'border': '#0F6E56'},
    {'x': 6.3, 'titulo': 'Etapa 3', 'subtitulo': 'ML clásico',
     'detalle': 'Ridge, Lasso,\nclustering,\nARIMA',
     'insight': 'R² ≈ 0.76\nen regresión',
     'color': '#F5C4B3', 'border': '#993C1D'},
    {'x': 9.2, 'titulo': 'Etapa 4', 'subtitulo': 'Redes neuronales',
     'detalle': 'MLP, LSTM\n+ comparación\ncon ML clásico',
     'insight': 'R² ≈ 0.93\nen regresión',
     'color': '#FAC775', 'border': '#854F0B'},
]

# Cajas y conexiones
for i, e in enumerate(etapas):
    # Caja principal
    box = FancyBboxPatch(
        (e['x'], 3.6), 2.3, 2.8,
        boxstyle='round,pad=0.05,rounding_size=0.15',
        facecolor=e['color'], edgecolor=e['border'], linewidth=1.5
    )
    ax.add_patch(box)
    ax.text(e['x']+1.15, 6.0, e['titulo'], ha='center', fontsize=11,
            fontweight='bold', color=e['border'])
    ax.text(e['x']+1.15, 5.5, e['subtitulo'], ha='center', fontsize=12,
            color=e['border'])
    ax.text(e['x']+1.15, 4.5, e['detalle'], ha='center', fontsize=9,
            color='#222', linespacing=1.4)

    # Insight abajo
    ax.text(e['x']+1.15, 3.0, e['insight'], ha='center', fontsize=9,
            style='italic', color='#444', linespacing=1.3)

    # Flecha hacia la siguiente
    if i < len(etapas) - 1:
        arrow = FancyArrowPatch(
            (e['x']+2.35, 5.0), (etapas[i+1]['x']-0.05, 5.0),
            arrowstyle='->', mutation_scale=22,
            color='#666', linewidth=1.3
        )
        ax.add_patch(arrow)

# Caja de cierre — recomendaciones
ax.text(6, 1.4,
        '↓  Insight central:  emisiones per cápita ≈ energía per cápita  (ρ ≈ 0.99)  ↓',
        ha='center', fontsize=11, color='#222', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#FAEEDA',
                  edgecolor='#854F0B', linewidth=1.2))
ax.text(6, 0.4,
        'Recomendaciones de política y aprendizajes metodológicos',
        ha='center', fontsize=10, style='italic', color='#555')

# Título principal
ax.text(6, 7.5, 'El viaje del proyecto — de 5 archivos crudos a recomendaciones',
        ha='center', fontsize=14, fontweight='bold', color='#222')

plt.tight_layout()
plt.show()

---

## 9. Recomendaciones finales

Cinco recomendaciones accionables que se desprenden del análisis:

**1. Comunicar siempre cuál "renovable" se está midiendo.** Cualquier reporte público sobre la matriz energética argentina debería separar la hidroeléctrica grande (matriz heredada) de las renovables modernas (ley 27.191). Mezclarlas oculta el verdadero estado de la transición.

**2. Priorizar políticas que actúen sobre energía per cápita.** Dado que `co2_per_capita ≈ 0.99 × energía per cápita`, las políticas climáticas de mayor impacto son las que reducen el consumo total de energía o cambian su mix de generación. Políticas sobre variables de segundo orden (eficiencia industrial puntual, transporte sectorial) tienen efecto más limitado.

**3. Para benchmarking internacional, usar comparables apropiados.** Comparar a Argentina con países de matriz, ingreso y geografía similares (Chile, Portugal, Brasil) da una lectura más útil que compararla con economías desarrolladas grandes (USA, Alemania, Japón).

**4. Construir series temporales más largas para mejorar el forecast.** El forecasting mensual de generación renovable se beneficiaría de más historia: con 96 meses, el techo de mejora era bajo. Si las series llegaran a 200+ meses, modelos como LSTM o Prophet podrían dar mejoras significativas.

**5. Para análisis recurrentes, preferir modelos auditables.** Cuando el modelo va a alimentar decisiones públicas o regulatorias, conviene Ridge o Lasso aunque tengan R² menor —la capacidad de explicar coeficientes a un comité no es opcional en ese contexto—.


---

## 10. Limitaciones del análisis

Un informe honesto incluye lo que **no** se puede afirmar con los datos disponibles. Cinco limitaciones importantes:

**El dataset global cubre 16 países, no el mundo entero.** Las conclusiones son válidas para esa muestra (que incluye los principales emisores y un buen mix de economías), pero generalizar al planeta entero requiere precauciones.

**La serie mensual de Argentina cubre 8 años (2011–2018).** Es una ventana corta para un sistema energético que tiene ciclos de inversión de 15+ años. Los modelos de forecast capturan el momento de transición, no la dinámica de largo plazo.

**El cambio estructural de 2018 no se anticipó por ningún modelo.** El boom del programa RenovAr fue una aceleración brusca que ningún modelo entrenado con datos previos pudo predecir. Esto refleja un límite real: **los modelos predicen la continuación del régimen pasado, no rupturas de política pública**.

**No se modelaron variables externas relevantes.** Precios internacionales de combustibles, variables climatológicas (años secos vs húmedos para hidro), costos de inversión en infraestructura. Cualquiera de ellas puede mover los resultados de un año a otro.

**Las correlaciones encontradas no son necesariamente causales.** Cuando decimos que las emisiones per cápita están explicadas por la energía per cápita, hablamos de asociación estadística — no demostramos que reducir una *cause* reducir la otra. Los modelos descriptivos no responden preguntas causales sin diseño experimental adicional.

---

## 11. Apéndice — Notebooks técnicos del proyecto

| Etapa | Notebook | Contenido |
|---|---|---|
| 1 | `01_exploracion_visual.ipynb` | EDA inicial: estadísticas descriptivas, visualizaciones, hallazgos. |
| 2 | `02_limpieza_transformacion.ipynb` | Pipeline ETL completo de las 5 fuentes. |
| 3 | `03_modelos_ml.ipynb` | Regresión lineal/Ridge/Lasso, clustering, ARIMA. |
| 4 | `04_redes_neuronales_paso4_1.ipynb` | MLP para regresión global. |
| 4 | `04_redes_neuronales_paso4_2.ipynb` | LSTM para forecasting. |
| 4 | `04_redes_neuronales_paso4_3.ipynb` | Tuning de la LSTM. |
| 4 | `04_redes_neuronales_paso4_4_A.ipynb` | Comparativa: regresión global. |
| 4 | `04_redes_neuronales_paso4_4_B.ipynb` | Comparativa: forecasting. |
| 4 | `04_redes_neuronales_paso4_4_C.ipynb` | Síntesis transversal y guía decisional. |
| 4 | `05_informe_final_storytelling.ipynb` | Este informe (storytelling final). |

Cada notebook es ejecutable de manera independiente y contiene su propia narrativa técnica. Este informe los toma como insumo y los traduce a un público no especializado.

*Notebook ejecutado para el Desafío Profesional de Data Science — Digital House.*